In [ ]:
pip install langchain_groq

In [ ]:
from google.colab import userdata

In [ ]:
from langgraph.graph import StateGraph , START , END
from langchain_groq import ChatGroq
from typing import TypedDict

In [ ]:
api_key = userdata.get("groq_api_key")
print(api_key)

[API KEY REDACTED]


In [ ]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=api_key
)

In [ ]:
class Cricket_state(TypedDict):
  runs:int
  balls:int
  four:int
  sixes:int

  strike_rate:float
  boundry_percent:float
  boundry_percent_ball:float
  summary:str

In [ ]:
# define strike rate percentage
def strike_rate(state:Cricket_state):
  run = state["runs"]
  ball = state["balls"]

  sr = run/ball*100
  # state["strike_rate"] = sr
  return {"strike_rate":sr}

In [ ]:
# define boundry percentage

def boundry_pct(state:Cricket_state):
  four = state["four"]
  sixes = state["sixes"]
  run = state["runs"]

  bp = (four*4)+(sixes*6)/run*100
  # state["boundry_percent"] = bp

  return {"boundry_percent":bp}

In [ ]:
# define boundry ball percentage

def boundry_ball_pct(state:Cricket_state):
  four = state["four"]
  sixes = state["sixes"]
  ball = state["balls"]

  bpb = four+sixes/ball


  return {"boundry_percent_ball":bpb}

In [ ]:
# define summary percentage

def summary(state:Cricket_state):
  summary = f"""
  Strike Rate : {state['strike_rate']} \n
  Boundary percent : {state['boundry_percent']}\n
  Balls per boundary : {state['boundry_percent_ball']}
  """
  return {"summary":summary}

In [ ]:
graph = StateGraph(Cricket_state)

# define nodes:
graph.add_node("Strike_rate" , strike_rate)
graph.add_node("boundry_pct" , boundry_pct)
graph.add_node("boundry_ball_pct" , boundry_ball_pct)
graph.add_node("summary" , summary)

#define edges:
graph.add_edge(START ,"Strike_rate")
graph.add_edge(START ,"boundry_pct")
graph.add_edge(START ,"boundry_ball_pct")

graph.add_edge("Strike_rate" , "summary" )
graph.add_edge("boundry_pct" , "summary" )
graph.add_edge("boundry_ball_pct" , "summary" )

workflow = graph.compile()

In [ ]:
initial_state = {
    "runs":100,
    "balls":50,
    "four":6,
    "sixes":4
}

In [ ]:
response = workflow.invoke(initial_state)
print(response["summary"])


  Strike Rate : 200.0 

  Boundary percent : 48.0

  Balls per boundary : 6.08
  
